# Lab 02 — Colunar e particionamento na prática (DuckDB)

**Onde roda:** 🟢 Browser (JupyterLite). DuckDB é um motor **colunar** — ótimo para sentir os conceitos.

Objetivo: escrever consultas que leem **só as colunas necessárias** e que **filtram pela coluna de partição** (data).

In [ ]:
try:
    import duckdb
except ModuleNotFoundError:
    import piplite; await piplite.install('duckdb'); import duckdb
con = duckdb.connect()
con.execute('CREATE TABLE fato_vendas(ano INT, mes INT, categoria VARCHAR, valor DOUBLE)')
con.executemany('INSERT INTO fato_vendas VALUES (?,?,?,?)', [
    (2023,12,'B',50.0),(2024,1,'A',100.0),(2024,2,'B',200.0),
    (2025,1,'A',300.0),(2025,1,'B',150.0),(2025,2,'A',250.0)])
con.execute('SELECT COUNT(*) AS linhas FROM fato_vendas').fetchone()

## 1. Leia só as colunas necessárias (evite SELECT *)
No colunar, pedir menos colunas = menos I/O. Aqui só `ano` e `valor`.

In [ ]:
con.execute('SELECT ano, valor FROM fato_vendas ORDER BY ano, valor').df()

## 2. Filtro pela coluna de partição (data) → pruning
Filtrar por `ano` é o padrão que, num DW particionado por data, faz o motor pular partições.

In [ ]:
con.execute('''
    SELECT categoria, SUM(valor) AS receita
    FROM fato_vendas
    WHERE ano = 2025
    GROUP BY categoria
    ORDER BY receita DESC
''').df()

## 3. Espiando o plano (EXPLAIN)
O `EXPLAIN` mostra como o motor vai executar — repare no filtro sendo aplicado cedo.

In [ ]:
print(con.execute("EXPLAIN SELECT SUM(valor) FROM fato_vendas WHERE ano = 2025").fetchall()[0][1])

## 4. Sua vez (mini-desafio)
Traga a **receita por ano** `(ano, receita)`, ordenada por `ano` crescente. Verifique.

In [ ]:
resposta = con.execute('''
    SELECT ano, SUM(valor) AS receita
    FROM fato_vendas
    GROUP BY ano
    ORDER BY ano
''').fetchall()
resposta

In [ ]:
def verificar(rows):
    esperado = [(2023,50.0),(2024,300.0),(2025,700.0)]
    try:
        assert rows == esperado, 'Agrupe por ano e some o valor.'
        print('✅ Correto! Agregação pela coluna de partição (ano).')
    except AssertionError as e:
        print('❌', e)

verificar(resposta)